# Ch12 - RLHF 概念演示

Reinforcement Learning from Human Feedback - ChatGPT 的核心技术

> 本 Notebook 配套《强化学习全面教程》PDF 使用。包含图形讲解单元。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle
import random
from collections import deque, defaultdict

# 中文字体配置
import os, platform
if platform.system() == 'Linux' and os.path.exists('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf'):
    fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'primary': '#887129', 'accent': '#50abc9', 'success': '#3a794f',
    'warning': '#94773b', 'error': '#a2524b', 'info': '#547da6',
    'muted': '#89867f', 'bg': '#f7f7f6', 'card': '#eeece9',
    'border': '#dad5c6', 'text': '#1f1e1c',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | 设备: {device} | Gymnasium: {gym.__version__}')


## 12.1 RLHF 三阶段

ChatGPT 等大模型对齐人类偏好的关键技术：

1. **SFT**：用指令数据微调基座模型
2. **RM 训练**：训练奖励模型
3. **PPO 微调**：用 RM 作为奖励，加 KL 惩罚

## 12.2 RLHF 完整流程图

In [ ]:
# 图形讲解：RLHF 三阶段流程
fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')

# 阶段 1: SFT
ax.add_patch(FancyBboxPatch((0.3, 4), 3.5, 2, boxstyle='round,pad=0.1',
    facecolor='#EFF6FF', edgecolor='#3B82F6', linewidth=2))
ax.text(2.05, 5.5, '阶段 1: SFT', ha='center', fontsize=13, fontweight='bold', color='#3B82F6')
ax.text(2.05, 4.8, '监督微调', ha='center', fontsize=10, color='#1F2937')
ax.text(2.05, 4.4, '基座 LLM + 指令数据', ha='center', fontsize=9, color=COLORS['muted'])

# 阶段 2: RM
ax.add_patch(FancyBboxPatch((5, 4), 3.5, 2, boxstyle='round,pad=0.1',
    facecolor='#FEF3C7', edgecolor='#F59E0B', linewidth=2))
ax.text(6.75, 5.5, '阶段 2: RM', ha='center', fontsize=13, fontweight='bold', color='#F59E0B')
ax.text(6.75, 4.8, '奖励模型训练', ha='center', fontsize=10, color='#1F2937')
ax.text(6.75, 4.4, '人类偏好对 (y_w, y_l)', ha='center', fontsize=9, color=COLORS['muted'])

# 阶段 3: PPO + KL
ax.add_patch(FancyBboxPatch((9.7, 4), 4, 2, boxstyle='round,pad=0.1',
    facecolor='#D1FAE5', edgecolor='#10B981', linewidth=2))
ax.text(11.7, 5.5, '阶段 3: PPO + KL', ha='center', fontsize=13, fontweight='bold', color='#10B981')
ax.text(11.7, 4.8, 'RL 微调 LLM', ha='center', fontsize=10, color='#1F2937')
ax.text(11.7, 4.4, 'r = RM - β·KL(π||π_SFT)', ha='center', fontsize=9, color=COLORS['muted'])

# 阶段间箭头
ax.annotate('', xy=(5, 5), xytext=(3.8, 5),
    arrowprops=dict(arrowstyle='->', color=COLORS['muted'], lw=2))
ax.annotate('', xy=(9.7, 5), xytext=(8.5, 5),
    arrowprops=dict(arrowstyle='->', color=COLORS['muted'], lw=2))

# 数据流
ax.text(4.4, 5.3, 'SFT 模型', fontsize=9, color='#3B82F6', fontweight='bold')
ax.text(8.6, 5.3, 'RM 模型', fontsize=9, color='#F59E0B', fontweight='bold')

# KL 惩罚说明
ax.add_patch(FancyBboxPatch((4, 1), 6, 2, boxstyle='round,pad=0.1',
    facecolor='#FEE2E2', edgecolor='#EF4444', linewidth=2))
ax.text(7, 2.5, 'KL 惩罚: 防止 reward hacking', ha='center', fontsize=12, fontweight='bold', color='#EF4444')
ax.text(7, 1.8, 'L = E[r_RM(x,y)] - β·KL[π_θ || π_SFT]', ha='center', fontsize=10, color='#1F2937', family='monospace')
ax.text(7, 1.3, 'β 太大: 策略不更新 | β 太小: reward hacking', ha='center', fontsize=9, color=COLORS['muted'])

# 连接到阶段 3
ax.annotate('', xy=(11.7, 3), xytext=(10, 3),
    arrowprops=dict(arrowstyle='->', color='#EF4444', lw=1.5, linestyle='dashed'))
ax.text(10.85, 3.3, '约束', fontsize=9, color='#EF4444')

ax.set_title('RLHF 三阶段流程 - ChatGPT 的核心技术', fontsize=14, fontweight='bold', pad=10)
plt.show()

## 12.3 KL 惩罚的作用

没有 KL 项，LLM 可能 reward hacking：学会输出 RM 给高分但无意义的内容。

In [ ]:
class ToyEnv:
    def __init__(self):
        self.n_states = 10; self.n_actions = 3; self.state = 0
    def reset(self):
        self.state = np.random.randint(0, self.n_states)
        return self.state
    def step(self, action):
        true_reward = 1.0 if action == 1 else 0.0
        self.state = (self.state + 1) % self.n_states
        return self.state, true_reward, False


class TinyPolicy(nn.Module):
    def __init__(self, n_states, n_actions):
        super().__init__()
        self.emb = nn.Embedding(n_states, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, s): return self.head(self.emb(s))


class RewardModel(nn.Module):
    def __init__(self, n_states, n_actions):
        super().__init__()
        self.emb = nn.Embedding(n_states, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, s): return self.head(self.emb(s))
    def hack_reward(self, s, a):
        return self.forward(s)[a] + (2.0 if a == 2 else 0.0)

def kl_divergence(logits_p, logits_q):
    p = torch.softmax(logits_p, dim=-1)
    log_p = torch.log_softmax(logits_p, dim=-1)
    log_q = torch.log_softmax(logits_q, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1).mean()

print('组件定义完成 ✓')

## 12.4 训练：对比不同 KL 系数

运行两组实验：β=0（无 KL 惩罚）和 β=0.5（有 KL 惩罚）。

In [ ]:
from torch.distributions import Categorical

def run_rlhf(kl_coef, steps=1500):
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    env = ToyEnv()
    policy = TinyPolicy(env.n_states, env.n_actions).to(device)
    ref_policy = TinyPolicy(env.n_states, env.n_actions).to(device)
    ref_policy.load_state_dict(policy.state_dict())
    reward_model = RewardModel(env.n_states, env.n_actions).to(device)
    # 预训练 RM
    rm_optim = optim.Adam(reward_model.parameters(), lr=1e-2)
    for _ in range(500):
        s = torch.randint(0, env.n_states, (32,))
        target = torch.tensor([0.0, 1.0, 2.0]).unsqueeze(0).expand(32, -1).to(device)
        pred = reward_model(s)
        loss = nn.functional.mse_loss(pred, target)
        rm_optim.zero_grad(); loss.backward(); rm_optim.step()
    # RLHF 训练
    p_optim = optim.Adam(policy.parameters(), lr=1e-3)
    action_dist = []
    for step in range(steps):
        s = env.reset()
        s_t = torch.tensor(s).to(device)
        logits = policy(s_t)
        dist = Categorical(logits=logits)
        a = dist.sample()
        log_prob = dist.log_prob(a)
        with torch.no_grad():
            rm_score = reward_model.hack_reward(s_t, a).item()
            ref_logits = ref_policy(s_t)
            kl = kl_divergence(logits.unsqueeze(0), ref_logits.unsqueeze(0)).item()
        shaped_reward = rm_score - kl_coef * kl
        with torch.no_grad():
            advantage = shaped_reward
        loss = -(log_prob * advantage).mean()
        loss = loss + kl_coef * kl_divergence(logits.unsqueeze(0), ref_logits.unsqueeze(0))
        p_optim.zero_grad(); loss.backward(); p_optim.step()
    # 采样最终动作分布
    counts = np.zeros(3)
    for _ in range(1000):
        s = env.reset()
        with torch.no_grad():
            a = policy(torch.tensor(s).to(device)).argmax().item()
        counts[a] += 1
    return counts / 1000

print('运行 β=0 (无 KL 惩罚)...')
dist_no_kl = run_rlhf(kl_coef=0.0)
print('运行 β=0.5 (有 KL 惩罚)...')
dist_with_kl = run_rlhf(kl_coef=0.5)
print(f'\n无 KL: action 0/1/2 = {dist_no_kl.round(3)}')
print(f'有 KL: action 0/1/2 = {dist_with_kl.round(3)}')

## 12.5 KL 惩罚效果对比

观察 β=0 与 β=0.5 时最终动作分布的差异。

In [ ]:
# 图形讲解：KL 惩罚效果对比
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

labels = ['action 0\n(无奖励)', 'action 1\n(真实偏好)', 'action 2\n(reward hacking)']
colors = ['#958249', '#3a794f', '#a2524b']

# 左：无 KL
ax = axes[0]
ax.bar(labels, dist_no_kl * 100, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
ax.set_ylabel('占比 (%)'); ax.set_ylim(0, 100)
ax.set_title('β=0 (无 KL 惩罚)\n→ reward hacking 失控', fontsize=11, fontweight='bold', color=COLORS['error'])
ax.grid(alpha=0.3, axis='y')
for i, v in enumerate(dist_no_kl * 100):
    ax.text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# 右：有 KL
ax = axes[1]
ax.bar(labels, dist_with_kl * 100, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
ax.set_ylabel('占比 (%)'); ax.set_ylim(0, 100)
ax.set_title('β=0.5 (有 KL 惩罚)\n→ 抑制 hacking, 接近真实偏好', fontsize=11, fontweight='bold', color=COLORS['success'])
ax.grid(alpha=0.3, axis='y')
for i, v in enumerate(dist_with_kl * 100):
    ax.text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('RLHF: KL 惩罚对 reward hacking 的抑制效果', fontsize=13, fontweight='bold')
plt.show()
print('\n观察:')
print('- β=0 时, 策略被 RM 误导, 选择了 action 2 (reward hacking)')
print('- β=0.5 时, KL 惩罚约束策略不偏离 SFT, 接近真实人类偏好 (action 1)')

## 12.6 小结

**RLHF 核心要点**：
- RM 可能被 reward hacking，必须用 KL 惩罚约束
- KL 系数 β 是关键超参数（见 12.5 对比图）
- 真实 LLM 场景还需处理长序列、奖励稀疏等挑战

**进阶方向**：
- **DPO**：跳过 RM，直接用偏好对优化
- **RLAIF**：用 AI 替代人类反馈
- **Constitutional AI**：让 AI 自我对齐



---

## 12.4 DPO: 直接偏好优化

RLHF 的三大痛点（需RM、PPO复杂、调参难）催生了 DPO。
DPO 证明了 RLHF 最优解有闭式表达，跳过 RM 和 PPO，一步完成对齐。

### DPO 数学推导（5步）

| 步骤 | 内容 |
|------|------|
| 第1步 | 写出 RLHF 目标: max E[r(x,y)] - β·KL[π‖π_ref] |
| 第2步 | 求闭式最优策略: π*(y|x) = π_ref(y|x)·exp(r/β)/Z(x) |
| 第3步 | 反解奖励: r(x,y) = β·log(π*/π_ref) + β·log Z(x) |
| 第4步 | 代入 Bradley-Terry 模型，Z(x) 在相减中消去 |
| 第5步 | DPO 损失: L = -E[log σ(β·(log(π_w/π_ref_w) - log(π_l/π_ref_l)))] |

**关键洞察**：不需要奖励模型、不需要 PPO，只需策略模型+参考模型+偏好对。

### 12.4.1 DPO vs RLHF 对比图

In [ ]:
# 图形讲解：RLHF 三阶段 vs DPO 单阶段
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# 左：RLHF 三阶段
ax = axes[0]
stages = ['SFT\n监督微调', 'RM\n奖励模型', 'PPO\n强化学习']
colors_rlhf = [COLORS['neg'], COLORS['accent'], COLORS['green']]
for i, (s, c) in enumerate(zip(stages, colors_rlhf)):
    ax.add_patch(FancyBboxPatch((0.5 + i*2.5, 1), 2, 1.5, boxstyle='round,pad=0.1',
                                 facecolor=c, edgecolor=c, linewidth=2))
    ax.text(1.5 + i*2.5, 1.75, s, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    if i < 2:
        ax.annotate('', xy=(0.5 + (i+1)*2.5, 1.75), xytext=(2.5 + i*2.5, 1.75),
                    arrowprops=dict(arrowstyle='->', color=COLORS['muted'], lw=2))
ax.set_xlim(0, 8); ax.set_ylim(0, 3.5); ax.axis('off')
ax.set_title('RLHF: 3阶段 + 4个模型', fontsize=13, fontweight='bold')

# 右：DPO 单阶段
ax = axes[1]
ax.add_patch(FancyBboxPatch((2.5, 1), 3, 1.5, boxstyle='round,pad=0.1',
                             facecolor=COLORS['accent'], edgecolor=COLORS['accent'], linewidth=2))
ax.text(4, 1.75, 'DPO\n直接偏好优化', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.set_xlim(0, 8); ax.set_ylim(0, 3.5); ax.axis('off')
ax.set_title('DPO: 1阶段 + 2个模型', fontsize=13, fontweight='bold')

fig.suptitle('RLHF vs DPO: 从 3 阶段到 1 阶段', fontsize=14, fontweight='bold')
plt.show()

### 12.4.2 DPO 从零实现（简化版）

用 toy 数据演示 DPO 损失计算过程，理解'好回答概率↑、坏回答概率↓'的核心思想。

In [ ]:
# DPO 简化实现：用 toy 数据演示核心机制
import torch.nn.functional as F

class SimpleDPO:
    '''简化 DPO：用 toy log-probs 演示损失计算'''
    def __init__(self, beta=0.1):
        self.beta = beta

    def loss(self, pi_w, pi_l, ref_w, ref_l):
        '''
        pi_w/pi_l: 策略模型对好/坏回答的 log prob
        ref_w/ref_l: 参考模型对好/坏回答的 log prob
        '''
        # DPO loss = -log σ(β * ((log(π_w/π_ref_w) - log(π_l/π_ref_l))))
        logits = self.beta * ((pi_w - ref_w) - (pi_l - ref_l))
        loss = -F.logsigmoid(logits)
        # 隐式奖励
        reward_w = self.beta * (pi_w - ref_w)
        reward_l = self.beta * (pi_l - ref_l)
        return loss, reward_w, reward_l

dpo = SimpleDPO(beta=0.5)

# 模拟 3 个训练步：策略逐渐学会偏好好回答
print('DPO 训练过程模拟（3步）')
print('-' * 60)
for step in range(3):
    # 初始：策略对好/坏回答概率差不多
    # 随训练进行，好回答概率逐渐升高
    pi_w = torch.tensor([-2.0 + step * 0.3])  # 好回答 log prob 上升
    pi_l = torch.tensor([-2.0 - step * 0.2])  # 坏回答 log prob 下降
    ref_w = torch.tensor([-2.0])  # 参考模型不变
    ref_l = torch.tensor([-2.0])
    loss, rw, rl = dpo.loss(pi_w, pi_l, ref_w, ref_l)
    print(f'步{step+1}: loss={loss.item():.4f} | reward_w={rw.item():.4f} reward_l={rl.item():.4f} | margin={rw.item()-rl.item():.4f}')

print('-' * 60)
print('观察：reward margin 逐渐增大 → 模型学会区分好/坏回答')

### 12.4.3 DPO vs RLHF 对比表

| 维度 | RLHF (PPO) | DPO | IPO | KTO |
|------|-----------|-----|-----|-----|
| 阶段数 | 3 (SFT+RM+PPO) | 1 | 1 | 1 |
| 模型数 | 4 (policy+ref+RM+value) | 2 (policy+ref) | 2 | 2 |
| 偏好数据 | 需要偏好对 | 需要偏好对 | 需要偏好对 | 只需好/坏标签 |
| 训练稳定性 | 低 (PPO调参难) | 高 (梯度下降) | 高 | 高 |
| 过拟合风险 | 中 | 高(小数据) | 低(L2正则) | 低 |
| 推荐场景 | 大规模 | 通用首选 | 小数据集 | 只有好/坏标签 |

### 12.4.4 DPO vs RLHF 训练曲线对比

模拟 RLHF 和 DPO 的训练过程，对比 reward margin 和训练稳定性。

In [ ]:
# 模拟 RLHF vs DPO 训练曲线对比
np.random.seed(42)
steps = 100

# 模拟 RLHF（高方差，有时不稳定）
rlhf_margin = np.cumsum(np.random.uniform(0.01, 0.04, steps)) + np.random.normal(0, 0.15, steps).cumsum()
rlhf_margin = np.maximum(rlhf_margin, 0)

# 模拟 DPO（平稳上升，低方差）
dpo_margin = np.cumsum(np.random.uniform(0.02, 0.03, steps)) + np.random.normal(0, 0.03, steps).cumsum()
dpo_margin = np.maximum(dpo_margin, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# 左：reward margin 曲线
ax = axes[0]
ax.plot(rlhf_margin, color=COLORS['pos'], lw=2, label='RLHF (PPO)', alpha=0.8)
ax.plot(dpo_margin, color=COLORS['accent'], lw=2, label='DPO')
ax.set_xlabel('训练步数'); ax.set_ylabel('Reward Margin')
ax.set_title('Reward Margin 对比', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

# 右：训练损失方差
ax = axes[1]
rlhf_loss_var = np.abs(np.random.normal(0, 0.3, steps)) + np.linspace(0.5, 0.2, steps)
dpo_loss_var = np.abs(np.random.normal(0, 0.05, steps)) + np.linspace(0.3, 0.05, steps)
ax.plot(rlhf_loss_var, color=COLORS['pos'], lw=2, label='RLHF (高方差)', alpha=0.8)
ax.plot(dpo_loss_var, color=COLORS['accent'], lw=2, label='DPO (低方差)')
ax.set_xlabel('训练步数'); ax.set_ylabel('Loss 标准差')
ax.set_title('训练稳定性对比', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('RLHF vs DPO: 训练稳定性与收敛速度', fontsize=14, fontweight='bold')
plt.show()

print('结论：DPO 训练更稳定（低方差），收敛更平滑；RLHF 偶尔出现不稳定波动。')

### 12.4.5 DPO 实战建议

- **beta=0.1** 最常用；学习率比 SFT 低 10 倍
- **参考模型**：用 SFT 后的模型，不用原始预训练模型
- **早停**：监控 reward margin，停止增大后即停训
- **混合 SFT**：训练中混入少量 SFT 数据，防止遗忘
- **TRL 库**：huggingface.co/trl 3 行代码启动 DPO 训练
- **实际应用**：Llama 3、Qwen 2、Mistral 等开源模型都用 DPO 对齐

---

## 12.5 总结

| 方法 | 核心思想 | 优势 | 劣势 |
|------|---------|------|------|
| RLHF | RM + PPO + KL惩罚 | 理论成熟 | 复杂、不稳定 |
| DPO | 闭式解跳过RM和PPO | 简洁、稳定 | 小数据易过拟合 |
| IPO | L2损失替代sigmoid | 防过拟合 | 需调β |
| KTO | 只需好/坏标签 | 数据需求减半 | 理论较新 |

**趋势**：从复杂到简洁，从需要大量数据到需要少量数据。DPO 已成为 LLM 对齐的事实标准。